In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
!pip install evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import evaluate
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# ── 1. Load data ──────────────────────────────────────────────
df = pd.read_csv('/kaggle/input/datasets/jamieteh/dataset-with-initial-threshold/Dataset_with_threshold.csv')
df = df[['Texts', 'Outputs']].dropna()

# Encode string labels to integers
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['Outputs'])
NUM_LABELS = len(le.classes_)
print(f"Total samples : {len(df)}")
print(f"Unique labels : {NUM_LABELS}")

# ── 2. Train / val split (80/20, fixed seed for fairness) ─────
train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label_id']
)
print(f"Train size: {len(train_df)} | Val size: {len(val_df)}")

# ── 3. Metric ─────────────────────────────────────────────────
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# ── 4. Models to compare ──────────────────────────────────────
models_to_compare = {
    "CiscoSecureBERT2.0": "cisco-ai/SecureBERT2.0-base",
    "SciBERT":             "allenai/scibert_scivocab_uncased"
}

results = {}

# ── 5. Training loop ──────────────────────────────────────────
for model_name, model_path in models_to_compare.items():
    print(f"\n{'='*50}")
    print(f"  Training: {model_name}")
    print(f"{'='*50}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        trust_remote_code=True
    )

    def tokenize(batch):
        return tokenizer(
            batch['Texts'],
            truncation=True,
            padding='max_length',
            max_length=128
        )

    train_dataset = Dataset.from_pandas(train_df[['Texts', 'label_id']].reset_index(drop=True))
    val_dataset   = Dataset.from_pandas(val_df[['Texts', 'label_id']].reset_index(drop=True))

    train_dataset = train_dataset.map(tokenize, batched=True)
    val_dataset   = val_dataset.map(tokenize, batched=True)

    train_dataset = train_dataset.rename_column('label_id', 'labels')
    val_dataset   = val_dataset.rename_column('label_id', 'labels')

    train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    val_dataset.set_format('torch',   columns=['input_ids', 'attention_mask', 'labels'])

    model = AutoModelForSequenceClassification.from_pretrained(
        model_path,
        num_labels=NUM_LABELS,
        ignore_mismatched_sizes=True,
        trust_remote_code=True
    )

    training_args = TrainingArguments(
        output_dir=f'./results_{model_name.replace(" ", "_")}',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy='epoch',
        save_strategy='no',
        logging_steps=50,
        load_best_model_at_end=False,
        seed=42,
        report_to='none'
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()

    eval_result = trainer.evaluate()
    acc = eval_result['eval_accuracy'] * 100
    results[model_name] = round(acc, 2)
    print(f"\n{model_name} final accuracy: {acc:.2f}%")

# ── 6. Final summary ──────────────────────────────────────────
print(f"\n{'='*50}")
print("  FINAL RESULTS")
print(f"{'='*50}")
for name, acc in results.items():
    print(f"  {name:25s}: {acc:.2f}%")
winner = max(results, key=results.get)
print(f"\n  Best model: {winner}")
print(f"  Accuracy gap: {abs(results['CiscoSecureBERT2.0'] - results['SciBERT']):.2f}%")

Total samples : 5089
Unique labels : 42
Train size: 4071 | Val size: 1018



  Training: CiscoSecureBERT2.0


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,1.423889,1.339402,0.696464
2,0.828554,0.896405,0.783890
3,0.489936,0.803742,0.802554



SciBERT final accuracy: 80.26%

  FINAL RESULTS
  CiscoSecureBERT2.0       : 87.33%
  SciBERT                  : 80.26%

  Best model: CiscoSecureBERT2.0
  Accuracy gap: 7.07%


In [3]:
!pip install evaluate -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
